# Phase 3 — Acquisition des données

**Autrice :** Faty Barbosa Monteiro

**Date :** 23 octobre 2026

**Client :** CLIENT001

**Lu après :** `documentation/02_data_architecture.ipynb`

L'architecture est fixée dans la phase 2. Cette note décrit le pipeline qui collecte ces données, les dépose telles que la source les a publiées, et charge dans la base les jeux prévus pour cela.


## 1. Objectif

Le pipeline est automatisé. Le registre des sources dit quel jeu existe, à quel rythme, et par quel programme. À chaque exécution, le calendrier décide si le jeu est dû. Un jeu dû est collecté. Un jeu qui ne l'est pas conserve le fichier déjà présent.

Le pipeline est écrit en Python. Il interroge les interfaces des sources, range les fichiers reçus et suit le calendrier du registre.


## 2. Marchés financiers

| Objet | Ce qui est collecté | Source | Moyen |
|---|---|---|---|
| Actions | Cours de clôture et volume de chaque séance | Yahoo Finance | Connecteur `yfinance` |
| ETF | Feuille des positions du fonds de comparaison, et rendement que ce fonds publie | iShares, fonds MSCI Global Sustainable Development Goals ETF | Téléchargement de la feuille. Le rendement publié est repris pour la comparaison en francs |
| Indices | Aucun cours | MSCI ACWI Sustainable Development Index (Net) | L'indice n'a pas de cours propre. Il n'entre pas dans le pipeline |

La feuille iShares ne contient que des actions. Le pipeline ne constitue pas un second univers de fonds cotés.


## 3. Éligibilité et exclusions

Les scores ESG ne sont pas collectés. Une note environnementale, sociale ou de gouvernance ne décide pas de l'entrée.

Les catégories d'Objectifs de développement durable sont le registre d'éligibilité. Pour chaque titre, il retient l'activité, les deux contributions, la preuve, la source, la controverse et le statut, saisis à partir des rapports de la société. Ce registre n'est pas un fichier de scores téléchargé chez un fournisseur.

Les exclusions du tabac, des armes conventionnelles, du charbon thermique et des jeux d'argent attendent une source nommée, société par société. Aucune collecte automatisée ne les invente.


## 4. Change et taux sans risque

| Donnée | Source | Moyen |
|---|---|---|
| Cours contre le franc | Banque nationale suisse | Connecteur `snb` |
| SARON | Banque nationale suisse | Connecteur `snb` |

`DS-004` demande au cube `devkum` chaque devise du référentiel que ce cube publie. Le cube ne publie pas `ILS`, `INR`, `SAR` ni `TWD`. Un titre dont la devise est absente de ce cube reste sans cours de change.


## 5. Jeux du registre

| Jeu | Source | Contenu | Moyen | Fréquence |
|---|---|---|---|---|
| `DS-000` | `SRC-001`, iShares | Feuille de positions du fonds | Téléchargement | Periodic |
| `DS-001` | `SRC-001` | Candidats tirés de cette feuille | Préparation | Periodic |
| `DS-002` | `SRC-001` | Référentiel des titres | Préparation | On demand |
| `DS-003` | `SRC-002`, Yahoo Finance | Cours de clôture et volume | Connecteur `yfinance` | Daily |
| `DS-004` | `SRC-003`, Banque nationale suisse | Cours de change | Connecteur `snb` | Monthly |
| `DS-005` | `SRC-003` | SARON | Connecteur `snb` | Monthly |
| `DS-006` | `SRC-004`, rapports de société | Registre ODD | Préparation. Le programme n'attribue pas le statut | Annual |

`DS-000` est le fichier reçu. Le chargement en publie les feuilles que les phases suivantes lisent : positions, rendement, historique de la valeur liquidative et distributions. Le document est du SpreadsheetML, donc chaque table part de son en-tête. Les autres jeux sont chargés aussi.

## 6. Pipeline

```text
Registre des sources
audit/source_registry.xlsx
        ↓
Jeu dû, selon la fréquence
        ↓
Connecteur ou téléchargement
src/acquisition/
        ↓
Fichier reçu
data/01_raw/
        ↓
Chargement des jeux autorisés
database/sdg_investment.db
        ↓
Journal
audit/acquisition_log.xlsx
```

La commande est `python -m src.pipeline.refresh`. Elle lit la fréquence de chaque jeu actif. Un jeu dû remplace le fichier reçu. L'ordre respecte les jeux dont un autre a besoin. L'argument `--force` collecte tous les jeux actifs sans regarder la fréquence. Un échec sur un jeu n'arrête pas les autres. La commande se termine alors avec un code d'erreur.

| Fréquence | Le jeu est dû |
|---|---|
| Daily | Aucune collecte réussie ce jour-là |
| Monthly | Aucune ce mois-ci |
| Periodic | Aucune ce mois-ci |
| Quarterly | Aucune ce trimestre |
| Annual | Aucune cette année |
| On demand | Le fichier n'existe pas encore |

Après cette collecte, la même commande charge la base, puis poursuit par les traitements des phases suivantes. Le périmètre de cette note est la collecte et le chargement.


## 7. Livrables

- les scripts de `src/acquisition/` et le chargement de `src/database/` ;
- la base `database/sdg_investment.db` ;
- le journal `audit/acquisition_log.xlsx` ;
- la documentation technique des connecteurs, `documentation/03_technical_documentation.ipynb` ;
- cette note.


## Dictionnaire

Ce dictionnaire reprend les termes employés dans cette note qui n'y sont pas déjà expliqués. Il n'en ajoute pas.

**API.** Interface de programmation par laquelle un connecteur interroge une source.

**Connecteur.** Programme qui lit une source et dépose le fichier reçu.

**Fréquence.** Rythme inscrit au registre pour un jeu : quotidien, mensuel, périodique, trimestriel, annuel, ou à la demande.

**Jeu dû.** Jeu que le calendrier demande de collecter lors de l'exécution.

**Score ESG.** Note environnementale, sociale ou de gouvernance diffusée par un fournisseur. Le pipeline ne la collecte pas.
